In [6]:
!apt update
!apt install -y tesseract-ocr
!apt install -y tesseract-ocr-tha # Installs the Thai language data
!apt install -y libtesseract-dev
!pip install pytesseract PyMuPDF pillow

'apt' is not recognized as an internal or external command,
operable program or batch file.
'apt' is not recognized as an internal or external command,
operable program or batch file.
'apt' is not recognized as an internal or external command,
operable program or batch file.
'apt' is not recognized as an internal or external command,
operable program or batch file.


   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.0 MB 12.5 MB/s eta 0:00:01
   ---------------------- ----------------- 3.9/7.0 MB 12.4 MB/s eta 0:00:01
   ------------------------------------- -- 6.6/7.0 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 11.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import fitz
import pytesseract
from pathlib import Path
from PIL import Image
import os

# Define words to detect - we include both the spelling provided and the standard dictionary spelling just in case!
TARGET_WORDS = [
    "รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฏร"
]

source_dir = Path('ลำปาง')
output_dir = Path('page-ocr')

# Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Find all PDF files
pdf_files = sorted(list(source_dir.glob('**/*.pdf')))
print(f"Found {len(pdf_files)} PDF files to process.")

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path}")
    
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"  Could not open {pdf_path}: {e}")
        continue
        
    found_pages = []
    
    # Iterate over all pages in the PDF
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        
        # Convert PDF page to high-res image (300 DPI)
        zoom = 300 / 72.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        # Extract text using Tesseract configured explicitly for Thai
        text = pytesseract.image_to_string(img, lang='tha')
        
        # Check for target words
        word_found = False
        for target in TARGET_WORDS:
            # Remove any hidden trailing whitespace when checking inside the text
            if target.strip() in text:
                word_found = True
                break
                
        if word_found:
            found_pages.append((page_idx + 1, text))
            print(f"  - Target word detected on page {page_idx + 1}")
            
    doc.close()
    
    # Save results to txt file if any matches found
    if found_pages:
        output_filename = pdf_path.stem + ".txt"
        output_path = output_dir / output_filename
        
        # Handle filename collisions
        if output_path.exists():
            output_filename = f"{pdf_path.parent.name}_{pdf_path.stem}.txt"
            output_path = output_dir / output_filename
            
        with open(output_path, 'w', encoding='utf-8') as f:
            for page_num, extracted_text in found_pages:
                f.write(f"--- Page {page_num} ---\n")
                f.write(extracted_text)
                f.write("\n\n")
        print(f"  Successfully saved matches to {output_path}")
    else:
        print(f"  No target word found in {pdf_path}")

print("Processing complete!")
